# Experiment 1 — Melamine NIR Spectroscopy

**Paper:** *Transferability Loss for Safe Model Selection under Domain Shift* (ICLR 2026)

**Dataset:** Melamine concentration in aqueous solutions measured by NIR spectroscopy across four preparation recipes (562, 568, 861, 862). Each recipe constitutes a distinct domain, yielding 12 pairwise source → target transfer tasks.

**Objective:** Evaluate TR-LOSS as a model-selection criterion for unsupervised domain adaptation, where the target-domain labels are unavailable at selection time.

**Protocol:**
1. For each of the 12 source–target pairs, fit KDA-PLS regression models over a grid of latent components (`n_comp`) and regularization strength (`λ`).
2. Compute the following quantities per model:
   - **RMSEV\_S** — source validation RMSE
   - **τ\_rMSE** (TR\_LOSS) — transferability loss between paired target-domain predictions
   - **RMSEV\_T** — target validation RMSE (oracle only)
   - **RMSEP\_T** — target test RMSE (final evaluation)
   - **MMD** — maximum mean discrepancy between source and target latent scores
3. Select models by minimising the composite criterion $J_{\mathrm{reg}} = \mathrm{MSE}_S + \tau_{\mathrm{MSE}}$ (Eq. 10) and compare against source-only, MMD-based, and oracle baselines.

## Load modules

In [1]:

import sys
sys.path.append("../../../methods/")

from kdaPLS.kdapls import KDAPLSRegression
from trloss import transferability_loss, selection_criterion
from metrics.mmd import mmd_rbf, mmd_linear


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mtply
import plotly.express as px
import plotly.graph_objects as go
import scipy.io as sp_io



from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


## Load Data

In [2]:
data_folder = '../../../data/melamine/'
data_file = "Melamine_Dataset.mat"

mat = sp_io.loadmat(data_folder + data_file, struct_as_record = True)

y_name = "y"
data_dict = {}

data_dict["X1_562"] = mat["Melamine_Dataset"][0][0][2][0][0][1] #swapped
data_dict["X2_562"] = mat["Melamine_Dataset"][0][0][2][0][0][0]
data_dict["Y_562"] = mat["Melamine_Dataset"][0][0][2][0][0][2]

data_dict["X1_568"] = mat["Melamine_Dataset"][0][0][3][0][0][0]
data_dict["X2_568"] = mat["Melamine_Dataset"][0][0][3][0][0][1]
data_dict["Y_568"] = mat["Melamine_Dataset"][0][0][3][0][0][2]

data_dict["X1_861"] = mat["Melamine_Dataset"][0][0][4][0][0][0]
data_dict["X2_861"] = mat["Melamine_Dataset"][0][0][4][0][0][1]
data_dict["Y_861"] = mat["Melamine_Dataset"][0][0][4][0][0][2]

data_dict["X1_862"] = mat["Melamine_Dataset"][0][0][5][0][0][0]
data_dict["X2_862"] = mat["Melamine_Dataset"][0][0][5][0][0][1]
data_dict["Y_862"] = mat["Melamine_Dataset"][0][0][5][0][0][2]

recipes = ["562","568","861","862"]

## Generate adaptations

In [3]:
# Hyperparameters
n_comp_list = [i for i in range(1,21)]  # Number of latent variables
lambda_list = np.logspace(0,10,10)      # Regularization parameter
lambda_list[0] = 0
kdict = {"type":"linear"}               # Kernel


In [4]:
model_results = {}
k = 0

for source_name in recipes:
    recipes_source = [re for re in recipes if re!=source_name]
    for target_name in recipes_source:
        print(source_name, target_name)

        Xs_all = data_dict[f"X2_{source_name}"]
        Xt_all = data_dict[f"X2_{target_name}"]
        ys_all = data_dict[f"Y_{source_name}"]
        yt_all = data_dict[f"Y_{target_name}"]

        # Splitting

        Xs, Xs_val_test, ys, ys_val_test = train_test_split(Xs_all, ys_all, test_size=0.8, random_state=42245368)
        Xs_val, Xs_test, ys_val, ys_test = train_test_split(Xs_val_test, ys_val_test, test_size=0.8, random_state=42245368)
        Xt, Xt_val_test, yt, yt_val_test = train_test_split(Xt_all, yt_all, test_size=0.8, random_state=42245368)
        Xt_val, Xt_test, yt_val, yt_test = train_test_split(Xt_val_test, yt_val_test, test_size=0.8, random_state=42245368)



        for n_comp in n_comp_list:
            
            for lambda_ in lambda_list:

                kdapls_model = KDAPLSRegression(xs = Xs, xt = Xt, n_components = n_comp, kdict = kdict, l=[lambda_], target_domain = 0)
                kdapls_model.fit(Xs,ys)
                # Validation and test prediction
                yt_val_pred_all = kdapls_model.predict_all(Xt_val)
                ys_val_pred = kdapls_model.predict(Xs_val)
                yt_test_pred = kdapls_model.predict(Xt_test)
                # Other distance metrics                
                ns = Xs.shape[0]
                nt = Xt.shape[0]
                mmd = mmd_linear(kdapls_model.x_scores_st_[:ns,:],kdapls_model.x_scores_st_[ns:,:])
                # Performance metrics
                tr_loss = np.sqrt(transferability_loss(yt_val_pred_all[0], yt_val_pred_all[1]))
                rmsev = np.sqrt(mean_squared_error(ys_val, ys_val_pred))
                r2v = r2_score(ys_val, ys_val_pred)
                rmsev_t = np.sqrt(mean_squared_error(yt_val, yt_val_pred_all[0]))
                r2v_t = r2_score(yt_val, yt_val_pred_all[0])
                rmsep_t = np.sqrt(mean_squared_error(yt_test, yt_test_pred))
                r2p_t = r2_score(yt_test, yt_test_pred)
                
                model_results[k] = {"TR_LOSS":tr_loss,
                                     "RMSEV_S":rmsev,
                                     "r2v":r2v,
                                    "RMSEV_T":rmsev_t,
                                      "r2v_t":r2v_t,
                                      "RMSEP_T":rmsep_t,
                                      "r2p_t":r2p_t,
                                      "n_comp":n_comp,
                                      "lambda_":lambda_ ,    
                                    "Source":source_name,
                                    "Target":target_name,
                                    "MMD":mmd
                                    

                                                         }
                k += 1
                


562 568
562 861
562 862
568 562
568 861
568 862
861 562
861 568
861 862
862 562
862 568
862 861


## Model selection


In [5]:
# Storing results in a DataFrame for easier analysis and plotting
model_results_pd = pd.DataFrame.from_dict(model_results, orient="index")
model_results_pd["rMSEV_S"] = model_results_pd["RMSEV_S"]
model_results_pd["rTR_LOSS"] = model_results_pd["TR_LOSS"]
model_results_pd["rMSEV_T"] = model_results_pd["RMSEV_T"]
model_results_pd["rMSEP_T"] = model_results_pd["RMSEP_T"]

# TR-LOSS criterion (Eq. 10 from paper): J_reg = MSE_S + τ_MSE  →  minimize
# TR_LOSS and RMSEV_S are in rMSE scale, so square them to get MSE
model_results_pd["RMSEV_S + TR_LOSS"] = selection_criterion(
    model_results_pd["RMSEV_S"]**2, model_results_pd["TR_LOSS"]**2, task="regression"
)

# MMD criterion: normalized per source-target pair (MMD not in MSE units)
modelsel_criterion_mmd = np.sqrt((model_results_pd["MMD"]/np.amax(model_results_pd["MMD"]))**2 + (model_results_pd["RMSEV_S"]/np.amax(model_results_pd["RMSEV_S"]))**2)
model_results_pd["RMSEV_S + MMD"] = modelsel_criterion_mmd

final_models_dict = {}
kk = 0
for source_name in recipes:
    recipes_source = [re for re in recipes if re!=source_name]
    for target_name in recipes_source:
        filter_domains = ((model_results_pd["Source"] == source_name) & (model_results_pd["Target"] == target_name))        
        model_result_st = model_results_pd.loc[filter_domains]
        final_models_dict[kk] = {
                                "Source":source_name,
                                "Target":target_name,
                                "RMSEV_S": model_result_st.sort_values("RMSEV_S").iloc[0]["RMSEP_T"],   
                                "RMSEV_S + MMD": model_result_st.sort_values("RMSEV_S + MMD").iloc[0]["RMSEP_T"],           
                                "RMSEV_S + TR_LOSS":model_result_st.sort_values("RMSEV_S + TR_LOSS").iloc[0]["RMSEP_T"],
                                "RMSEV_T": model_result_st.sort_values("RMSEV_T").iloc[0]["RMSEP_T"],}
        
        kk += 1
        
final_models_pd = pd.DataFrame.from_dict(final_models_dict, orient="index")
# final_models_pd.to_csv("../output/01_melamine_dapls.csv")
print(final_models_pd.to_latex(index=False,float_format="{:.2f}".format))


\begin{tabular}{llrrrr}
\toprule
Source & Target & RMSEV_S & RMSEV_S + MMD & RMSEV_S + TR_LOSS & RMSEV_T \\
\midrule
562 & 568 & 2.03 & 2.54 & 2.11 & 1.99 \\
562 & 861 & 4.10 & 4.10 & 2.78 & 2.52 \\
562 & 862 & 8.94 & 2.50 & 2.50 & 2.44 \\
568 & 562 & 2.41 & 2.41 & 2.47 & 2.41 \\
568 & 861 & 2.83 & 2.92 & 4.06 & 2.59 \\
568 & 862 & 4.33 & 2.60 & 2.81 & 2.48 \\
861 & 562 & 27.79 & 5.13 & 2.62 & 2.57 \\
861 & 568 & 17.49 & 3.29 & 2.44 & 2.41 \\
861 & 862 & 9.81 & 2.60 & 2.61 & 2.60 \\
862 & 562 & 33.09 & 4.65 & 4.50 & 3.38 \\
862 & 568 & 36.37 & 3.21 & 3.71 & 2.92 \\
862 & 861 & 10.94 & 8.06 & 4.17 & 3.47 \\
\bottomrule
\end{tabular}



## Visualize selections

In [6]:
model_results_862 = model_results_pd.loc[model_results_pd["Source"] == "862"].copy()

fig = px.scatter(model_results_862, x="rTR_LOSS", y="rMSEV_S", color="rMSEP_T",
                 hover_data=["n_comp", "lambda_"],
                 facet_col="Target", range_color=[3, 15],
                 category_orders={"Target": ["562", "568", "861"]},
                 color_continuous_scale=px.colors.sequential.Viridis_r,
                 labels={"rTR_LOSS": "τ<sub>rMSE</sub>"})
fig.update_traces(marker=dict(size=10))

# Overlay red cross for TR_LOSS-selected model, blue square for oracle (best RMSEP_T)
for i, target_name in enumerate(["562", "568", "861"]):
    mask = model_results_862["Target"] == target_name
    subset = model_results_862.loc[mask]
    best_trloss = subset.loc[subset["RMSEV_S + TR_LOSS"].idxmin()]
    best_oracle = subset.loc[subset["RMSEP_T"].idxmin()]

    col_idx = i + 1

    # Selected model (TR_LOSS criterion)
    fig.add_trace(
        go.Scatter(
            x=[best_trloss["rTR_LOSS"]],
            y=[best_trloss["rMSEV_S"]],
            mode="markers",
            marker=dict(symbol="cross", size=12, color="red"),
            name="TR-Loss (τ)",
            legendgroup="selected",
            showlegend=(i == 0),
            hovertemplate=(f"Target={target_name}<br>"
                           f"n_comp={best_trloss['n_comp']}, λ={best_trloss['lambda_']:.1f}<br>"
                           f"rMSEP_T={best_trloss['rMSEP_T']:.2f}<extra></extra>")
        ),
        row=1, col=col_idx
    )

    # Oracle model (lowest RMSEP_T)
    fig.add_trace(
        go.Scatter(
            x=[best_oracle["rTR_LOSS"]],
            y=[best_oracle["rMSEV_S"]],
            mode="markers",
            marker=dict(symbol="square", size=12, color="blue", line=dict(width=1, color="black")),
            name="Oracle (best RMSEP<sub>T</sub>)",
            legendgroup="oracle",
            showlegend=(i == 0),
            hovertemplate=(f"Target={target_name}<br>"
                           f"n_comp={best_oracle['n_comp']}, λ={best_oracle['lambda_']:.1f}<br>"
                           f"rMSEP_T={best_oracle['rMSEP_T']:.2f}<extra></extra>")
        ),
        row=1, col=col_idx
    )

fig.update_layout(
    width=1000, height=300, font_size=18, title="Source=862",
    legend=dict(
        yanchor="top", y=0.95,
        xanchor="left", x=0.77,
        bgcolor="rgba(255,255,255,0.8)",
        font=dict(size=11)
    )
)
fig.update_xaxes(range=[0, 41])
fig.update_yaxes(range=[0, 6])
fig.show()

### All pairs

In [7]:
# Build DataFrames of selected + oracle models for overlay
plot_df = model_results_pd.copy()

selected_rows = []
oracle_rows = []
for source_name in recipes:
    for target_name in [r for r in recipes if r != source_name]:
        mask = (plot_df["Source"] == source_name) & (plot_df["Target"] == target_name)
        subset = plot_df.loc[mask]
        selected_rows.append(subset.loc[subset["RMSEV_S + TR_LOSS"].idxmin()])
        oracle_rows.append(subset.loc[subset["RMSEP_T"].idxmin()])

selected_df = pd.DataFrame(selected_rows)
oracle_df = pd.DataFrame(oracle_rows)

# ---------- Plot 1: TRLoss space ----------
fig_tr = px.scatter(
    plot_df, x="rTR_LOSS", y="rMSEV_S", color="rMSEP_T",
    hover_data=["n_comp", "lambda_", "Source", "Target"],
    facet_row="Source", facet_col="Target",
    category_orders={"Source": recipes, "Target": recipes},
    range_color=[3, 15],
    color_continuous_scale=px.colors.sequential.Viridis_r,
    labels={"rTR_LOSS": "τ_rMSE"},
)
# Fade background points so overlays stand out
n_bg_traces = len(fig_tr.data)
fig_tr.update_traces(marker=dict(size=7, opacity=0.4))

# Overlay selected models (added AFTER background → rendered on top)
fig_sel = px.scatter(
    selected_df, x="rTR_LOSS", y="rMSEV_S",
    facet_row="Source", facet_col="Target",
    category_orders={"Source": recipes, "Target": recipes},
)
for i, trace in enumerate(fig_sel.data):
    trace.marker = dict(symbol="cross", size=14, color="red", opacity=1.0, line=dict(width=2, color="darkred"))
    trace.name = "TR-Loss (τ)"
    trace.legendgroup = "selected_tr"
    trace.showlegend = (i == 0)
    fig_tr.add_trace(trace)

# Overlay oracle models (added last → on the very top)
fig_ora = px.scatter(
    oracle_df, x="rTR_LOSS", y="rMSEV_S",
    facet_row="Source", facet_col="Target",
    category_orders={"Source": recipes, "Target": recipes},
)
for i, trace in enumerate(fig_ora.data):
    trace.marker = dict(symbol="square", size=12, color="blue", opacity=1.0, line=dict(width=1.5, color="black"))
    trace.name = "Oracle (best RMSEP<sub>T</sub>)"
    trace.legendgroup = "oracle_tr"
    trace.showlegend = (i == 0)
    fig_tr.add_trace(trace)

fig_tr.update_layout(width=1050, height=900, font_size=12, title="TR-Loss selection vs. Oracle",
                     legend=dict(
        yanchor="top", y=0.975,
        xanchor="left", x=0.025,
        bgcolor="rgba(255,255,255,0.8)",
        font=dict(size=11)))

fig_tr.update_xaxes(range=[0, 41])
fig_tr.update_yaxes(range=[0, 6])
fig_tr.show()